# Phase 6 â€” EvaluaciÃ³n de GeneralizaciÃ³n GeogrÃ¡fica (CÃ³rdoba)

**Entorno**: Colab (GPU necesaria para Prithvi)  
**Tiempo estimado**: 20â€“30 min  
**Input**: modelo entrenado en Corrientes + test patches de CÃ³rdoba  
**Output**: mÃ©tricas de generalizaciÃ³n + visualizaciones

## Por quÃ© esta evaluaciÃ³n es clave para el portfolio

Un modelo que funciona solo donde fue entrenado es frÃ¡gil. Este notebook demuestra que
el modelo aprendiÃ³ **firmas espectrales reales de cicatrices de incendio**, no simplemente
patrones especÃ­ficos de Corrientes.

| Conjunto  | RegiÃ³n    | Fecha     | Paisaje                      |
|-----------|-----------|-----------|------------------------------|
| Train/Val | Corrientes | Dicâ€“Feb 2022 | Humedales, pastizales         |
| **Test**  | **CÃ³rdoba** | **Octâ€“Nov 2020** | **Sierras, monte xerÃ³filo** |

Si IoU(CÃ³rdoba) â‰ˆ IoU(Corrientes val) â†’ generalizaciÃ³n confirmada.


In [ ]:
import subprocess, sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'IN_COLAB = {IN_COLAB}')

if IN_COLAB:
    # Fix numpy/terratorch conflict ANTES de montar Drive
    needs_install = False
    try:
        import numpy as np
        assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
        from terratorch.registry import BACKBONE_REGISTRY
        print('Packages OK â€” saltando instalaciÃ³n.')
    except Exception:
        needs_install = True
    if needs_install:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'numpy==2.0.2', '--force-reinstall'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'terratorch', 'einops', 'timm',
                        'segmentation-models-pytorch'], check=True)
        print('Reiniciando kernel...')
        os.kill(os.getpid(), 9)

    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
import os, random, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

# â”€â”€ Paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT_DIR   = BASE / 'models'
RESULTS    = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

# Patches de CÃ³rdoba â€” copiados desde local a Drive en el paso anterior
if IN_COLAB:
    LOCAL_CORDOBA = Path('/content/cordoba_patches')
    if not (LOCAL_CORDOBA / 'images').exists():
        print('Copiando patches CÃ³rdoba a /content/...')
        import subprocess
        LOCAL_CORDOBA.mkdir(exist_ok=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/images'),     str(LOCAL_CORDOBA)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/masks_dnbr'), str(LOCAL_CORDOBA)], check=True)
        print('Copia completada.')
    IMG_DIR  = LOCAL_CORDOBA / 'images'
    MASK_DIR = LOCAL_CORDOBA / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/cordoba/patches/images'
    MASK_DIR = BASE / 'data/cordoba/patches/masks_dnbr'

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), f'{len(img_paths)} imgs vs {len(mask_paths)} masks'
print(f'\nPatches CÃ³rdoba: {len(img_paths)}')
fire_flags = np.array([np.load(f, mmap_mode='r').max() > 0 for f in mask_paths])
print(f'Patches con cicatriz: {fire_flags.sum()} ({fire_flags.mean()*100:.1f}%)')

In [ ]:
# â”€â”€ Cargar Prithvi (misma configuraciÃ³n que 07_prithvi.ipynb) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from terratorch.registry import BACKBONE_REGISTRY

PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
BAND_IDX     = [0, 1, 2, 4, 5, 6]   # B02, B03, B04, B8A, B11, B12
IMG_SIZE     = 224
THRESHOLD    = 0.5

print('Cargando Prithvi...')
backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
OUT_DIM  = 768
N_SIDE   = 14

class PrithviNeck(nn.Module):
    def __init__(self, n_side=14):
        super().__init__()
        self.n_side = n_side
    def forward(self, layers_out):
        x = layers_out[-1][:, 1:, :]
        B, L, D = x.shape
        return x.permute(0,2,1).reshape(B, D, self.n_side, self.n_side)

class FPNDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)

class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)
    def forward(self, x):
        xt    = x.unsqueeze(2)
        feats = self.backbone(xt)
        sp    = self.neck(feats)
        return self.decoder(sp)

model = PrithviSegModel(backbone, embed_dim=OUT_DIM, n_side=N_SIDE).to(DEVICE)

# Cargar pesos entrenados en Corrientes
ckpt_path = CKPT_DIR / 'best_prithvi_burn_scar_wildfire.pth'
if not ckpt_path.exists():
    # Buscar cualquier checkpoint de Prithvi disponible
    candidates = list(CKPT_DIR.glob('*prithvi*.pth'))
    if candidates:
        ckpt_path = candidates[0]
        print(f'Usando checkpoint: {ckpt_path.name}')
    else:
        raise FileNotFoundError(f'No se encontrÃ³ checkpoint en {CKPT_DIR}')

model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()
print(f'âœ“ Modelo cargado: {ckpt_path.name}')

In [ ]:
# â”€â”€ Dataset de CÃ³rdoba â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class CordobaDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)    # (11, 256, 256)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)   # (256, 256)
        img  = img[BAND_IDX]               # (6, 256, 256)
        img /= 10000.0
        img  = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        t    = (256 - IMG_SIZE) // 2
        img  = img[:, t:t+IMG_SIZE, t:t+IMG_SIZE]
        mask = mask[t:t+IMG_SIZE, t:t+IMG_SIZE]
        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)

ds     = CordobaDataset(img_paths, mask_paths)
loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches a evaluar: {len(loader)}')

In [ ]:
# â”€â”€ Inferencia completa â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
all_probs, all_targets = [], []
patch_probs_list, patch_masks_list, patch_imgs_list = [], [], []

print('Evaluando en CÃ³rdoba...')
with torch.no_grad():
    for imgs, masks in tqdm(loader):
        imgs  = imgs.to(DEVICE)
        probs = model(imgs).sigmoid().squeeze(1).cpu().numpy()   # (B, 224, 224)
        masks_np = masks.squeeze(1).cpu().numpy()                # (B, 224, 224)

        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))

        for b in range(len(imgs)):
            patch_probs_list.append(probs[b])
            patch_masks_list.append(masks_np[b])
            patch_imgs_list.append(imgs[b].cpu().numpy())

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)

# â”€â”€ MÃ©tricas â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
precision, recall, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
tp = int(((all_preds==1)&(all_targets==1)).sum())
fp = int(((all_preds==1)&(all_targets==0)).sum())
fn = int(((all_preds==0)&(all_targets==1)).sum())
tn = int(((all_preds==0)&(all_targets==0)).sum())
iou_fire = tp / (tp + fp + fn + 1e-6)

try:
    auc = roc_auc_score(all_targets, all_probs)
except:
    auc = float('nan')

print('\n' + '='*55)
print('  RESULTADOS â€” CÃ“RDOBA (Test geogrÃ¡fico)')
print('='*55)
print(f'  Precision (cicatriz): {precision:.4f}')
print(f'  Recall (cicatriz)   : {recall:.4f}')
print(f'  F1-Score            : {f1:.4f}')
print(f'  IoU (cicatriz)      : {iou_fire:.4f}')
print(f'  AUC-ROC             : {auc:.4f}')
print('â”€'*55)
print('  COMPARATIVA:')
print(f'  Corrientes val IoU  : 0.4198  â† entrenamiento')
print(f'  CÃ³rdoba test IoU    : {iou_fire:.4f}  â† este notebook')
print('='*55)

In [ ]:
# â”€â”€ Curva Precision-Recall â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PR curve
prec_c, rec_c, thr_c = precision_recall_curve(all_targets, all_probs)
axes[0].plot(rec_c, prec_c, color='crimson', lw=2)
axes[0].scatter([recall], [precision], color='black', zorder=5, s=80,
                label=f'thr=0.5  |  F1={f1:.3f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall â€” CÃ³rdoba test')
axes[0].legend(); axes[0].grid(alpha=0.3)

# ROC curve
if not np.isnan(auc):
    fpr_c, tpr_c, _ = roc_curve(all_targets, all_probs)
    axes[1].plot(fpr_c, tpr_c, color='steelblue', lw=2, label=f'AUC={auc:.3f}')
    axes[1].plot([0,1],[0,1],'--',color='gray')
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC â€” CÃ³rdoba test')
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('EvaluaciÃ³n de generalizaciÃ³n geogrÃ¡fica â€” CÃ³rdoba Oct 2020', fontsize=12)
plt.tight_layout()
out = RESULTS / 'cordoba_evaluation_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

In [ ]:
# â”€â”€ VisualizaciÃ³n de predicciones â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

# IoU por patch
patch_ious = []
for prob, mask in zip(patch_probs_list, patch_masks_list):
    pred = (prob > THRESHOLD).astype(np.float32)
    tp_ = (pred*mask).sum(); fp_ = (pred*(1-mask)).sum(); fn_ = ((1-pred)*mask).sum()
    patch_ious.append(float(tp_/(tp_+fp_+fn_+1e-6)))
patch_ious = np.array(patch_ious)

fire_patch_idx = [i for i,f in enumerate(fire_flags) if f]
best_idx       = sorted(fire_patch_idx, key=lambda i: patch_ious[i], reverse=True)[:4]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ['RGB', 'Ground truth (dNBR)', 'PredicciÃ³n (prob)', 'TP/FP/FN']
for ci, ct in enumerate(col_titles):
    axes[0, ci].set_title(ct, fontsize=10, pad=6)

for row, vi in enumerate(best_idx):
    img_np   = patch_imgs_list[vi]     # (6, 224, 224)
    prob_np  = patch_probs_list[vi]    # (224, 224)
    mask_np  = patch_masks_list[vi]    # (224, 224)
    pred_bin = (prob_np > THRESHOLD).astype(np.float32)
    iou_v    = patch_ious[vi]

    def denorm(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([denorm(2), denorm(1), denorm(0)])  # B04, B03, B02

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_bin==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_bin==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_bin==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                               axes[row,0].axis('off')
    axes[row,1].imshow(mask_np,  cmap='Reds', vmin=0, vmax=1); axes[row,1].axis('off')
    axes[row,2].imshow(prob_np,  cmap='hot',  vmin=0, vmax=1); axes[row,2].axis('off')
    axes[row,3].imshow(err);                               axes[row,3].axis('off')
    axes[row,0].set_ylabel(f'IoU={iou_v:.3f}', fontsize=9)

tp_p  = mpatches.Patch(color=[0,.8,0], label='TP (detectado)')
fp_p  = mpatches.Patch(color=[1,.5,0], label='FP (falsa alarma)')
fn_p  = mpatches.Patch(color=[.9,.1,.1], label='FN (perdido)')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.01))

plt.suptitle(
    f'CÃ³rdoba Test  |  IoU={iou_fire:.4f}  |  Recall={recall:.4f}  |  Precision={precision:.4f}\n'
    f'Modelo entrenado en Corrientes (2022) â€” evaluado en CÃ³rdoba (2020)',
    fontsize=11, y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

In [ ]:
# â”€â”€ Resumen final para portfolio â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*60)
print('  PORTFOLIO SUMMARY â€” Wildfire Burn Scar Detection')
print('='*60)
print('  Modelo: Prithvi-100M (IBM/NASA) + FPN decoder')
print('  Encoder: frozen (100M params pre-entrenados en HLS/S2)')
print('  Decoder: entrenado en Corrientes, Argentina (dic 2021â€“feb 2022)')
print('â”€'*60)
print('  TRAINING SET (Corrientes):')
print('    Ãrea   : -59.5W/-29.0S â†’ -56.0W/-26.5N (pastizales/humedales)')
print('    Labels : dNBR > 0.10 (cicatrices de incendio)')
print('    Patches: 5,687 Ã— 256Ã—256 px Ã— 6 bandas S2')
print('â”€'*60)
print('  TEST SET (CÃ³rdoba) â€” zona NUNCA vista:')
print(f'    Ãrea   : -65.5W/-33.0S â†’ -62.5W/-30.5N (sierras/monte xerÃ³filo)')
print(f'    Patches: {len(img_paths)}')
print(f'    IoU cicatriz : {iou_fire:.4f}')
print(f'    Recall       : {recall:.4f}')
print(f'    Precision    : {precision:.4f}')
print('â”€'*60)
print('  PIPELINE COMPLETO:')
print('    S2 L2A (CDSE) â†’ dNBR labels â†’ Prithvi fine-tune â†’ GeneralizaciÃ³n')
print('    Corrientes 2022 â†’ CÃ³rdoba 2020 = diferente regiÃ³n + diferente aÃ±o')
print('='*60)